In [1]:
import pandas as pd
import ast
from itertools import combinations


In [2]:
# Read the CSV file
df = pd.read_csv('../../../../data/metrics/experiments/system_level_comparison/systems_data_with_metrics-with_penalty.csv')
print(f"Total rows: {len(df)}")
print(f"Systems: {df['system'].unique()}")
print(f"Unique questions: {df['question_id'].nunique()}")
df.head()


Total rows: 4601
Systems: ['chess' 'din-sql' 'mac-sql']
Unique questions: 1534


,system,db_name,question_id,question,gold_sql,predicted_sql,EXECUTION_ACCURACY,EXACT_COLUMN_AND_EXACT_CELL,EXACT_COLUMN_AND_PARTIAL_CELL,SEMANTIC_COLUMN_AND_EXACT_CELL,SEMANTIC_COLUMN_AND_PARTIAL_CELL,NO_COLUMN_AND_PARTIAL_CELL
0,chess,california_schools,0,What is the highest eligible free rate for K-1...,SELECT `Free Meal Count (K-12)` / `Enrollment ...,SELECT MAX(`Free Meal Count (K-12)` / `Enrollm...,"{""EX"": 1, ""latency"": 0.03451204299926758}","{""EX"": 1, ""EXP"": 1.0, ""EXR"": 1.0, ""F1"": 1.0, ""...","{""EX"": 1, ""EXP"": 1.0, ""EXR"": 1.0, ""F1"": 1.0, ""...","{""EX"": 1, ""EXP"": 1.0, ""EXR"": 1.0, ""F1"": 1.0, ""...","{""EX"": 1, ""EXP"": 1.0, ""EXR"": 1.0, ""F1"": 1.0, ""...","{""EX"": 1, ""EXP"": 1.0, ""EXR"": 1.0, ""F1"": 1.0, ""..."
1,chess,california_schools,1,Please list the lowest three eligible free rat...,SELECT `Free Meal Count (Ages 5-17)` / `Enroll...,SELECT (`Free Meal Count (Ages 5-17)` / `Enrol...,"{""EX"": 1, ""latency"": 0.017229080200195312}","{""EX"": 1, ""EXP"": 1.0, ""EXR"": 1.0, ""F1"": 1.0, ""...","{""EX"": 1, ""EXP"": 1.0, ""EXR"": 1.0, ""F1"": 1.0, ""...","{""EX"": 1, ""EXP"": 1.0, ""EXR"": 1.0, ""F1"": 1.0, ""...","{""EX"": 1, ""EXP"": 1.0, ""EXR"": 1.0, ""F1"": 1.0, ""...","{""EX"": 1, ""EXP"": 1.0, ""EXR"": 1.0, ""F1"": 1.0, ""..."
2,chess,california_schools,2,Please list the zip code of all the charter sc...,SELECT T2.Zip FROM frpm AS T1 INNER JOIN schoo...,SELECT DISTINCT S.Zip FROM frpm AS F INNER JOI...,"{""EX"": 1, ""latency"": 0.006277799606323242}","{""EX"": 1, ""EXP"": 1.0, ""EXR"": 1.0, ""F1"": 1.0, ""...","{""EX"": 1, ""EXP"": 1.0, ""EXR"": 1.0, ""F1"": 1.0, ""...","{""EX"": 1, ""EXP"": 1.0, ""EXR"": 1.0, ""F1"": 1.0, ""...","{""EX"": 1, ""EXP"": 1.0, ""EXR"": 1.0, ""F1"": 1.0, ""...","{""EX"": 1, ""EXP"": 1.0, ""EXR"": 1.0, ""F1"": 1.0, ""..."
3,chess,california_schools,3,What is the unabbreviated mailing street addre...,SELECT T2.MailStreet FROM frpm AS T1 INNER JOI...,SELECT T2.MailStreet FROM frpm AS T1 INNER JOI...,"{""EX"": 1, ""latency"": 0.009787797927856445}","{""EX"": 1, ""EXP"": 1.0, ""EXR"": 1.0, ""F1"": 1.0, ""...","{""EX"": 1, ""EXP"": 1.0, ""EXR"": 1.0, ""F1"": 1.0, ""...","{""EX"": 1, ""EXP"": 1.0, ""EXR"": 1.0, ""F1"": 1.0, ""...","{""EX"": 1, ""EXP"": 1.0, ""EXR"": 1.0, ""F1"": 1.0, ""...","{""EX"": 1, ""EXP"": 1.0, ""EXR"": 1.0, ""F1"": 1.0, ""..."
4,chess,california_schools,4,Please list the phone numbers of the direct ch...,SELECT T2.Phone FROM frpm AS T1 INNER JOIN sch...,--error--,"{""EX"": 0, ""latency"": 0.001291036605834961}","{""EXP"": 0.0, ""EXR"": 0.0, ""F1"": 0.0, ""EX"": 0, ""...","{""EXP"": 0.0, ""EXR"": 0.0, ""F1"": 0.0, ""EX"": 0, ""...","{""EXP"": 0.0, ""EXR"": 0.0, ""F1"": 0.0, ""EX"": 0, ""...","{""EXP"": 0.0, ""EXR"": 0.0, ""F1"": 0.0, ""EX"": 0, ""...","{""EXP"": 0.0, ""EXR"": 0.0, ""F1"": 0.0, ""EX"": 0, ""..."


In [3]:
# Parse metric dictionary safely
def parse_metric_value(x):
    """Safely parse a metric value from string to dict"""
    if pd.isna(x) or x == '' or x == 'nan':
        return None
    try:
        x_str = str(x) if not isinstance(x, str) else x
        return ast.literal_eval(x_str)
    except (ValueError, SyntaxError):
        return None

# Extract EX, EXP, EXR from SEMANTIC_COLUMN_AND_EXACT_CELL
metric_col = 'SEMANTIC_COLUMN_AND_EXACT_CELL'
df[f'{metric_col}_EX'] = df[metric_col].apply(
    lambda x: parse_metric_value(x)['EX'] if parse_metric_value(x) else None
)
df[f'{metric_col}_EXP'] = df[metric_col].apply(
    lambda x: parse_metric_value(x)['EXP'] if parse_metric_value(x) else None
)
df[f'{metric_col}_EXR'] = df[metric_col].apply(
    lambda x: parse_metric_value(x)['EXR'] if parse_metric_value(x) else None
)

print("\nParsed metrics successfully")
print(f"Non-null EX values: {df[f'{metric_col}_EX'].notna().sum()}")
df[[metric_col, f'{metric_col}_EX', f'{metric_col}_EXP', f'{metric_col}_EXR']].head()



Parsed metrics successfully
Non-null EX values: 4597


,SEMANTIC_COLUMN_AND_EXACT_CELL,SEMANTIC_COLUMN_AND_EXACT_CELL_EX,SEMANTIC_COLUMN_AND_EXACT_CELL_EXP,SEMANTIC_COLUMN_AND_EXACT_CELL_EXR
0,"{""EX"": 1, ""EXP"": 1.0, ""EXR"": 1.0, ""F1"": 1.0, ""...",1.0,1.0,1.0
1,"{""EX"": 1, ""EXP"": 1.0, ""EXR"": 1.0, ""F1"": 1.0, ""...",1.0,1.0,1.0
2,"{""EX"": 1, ""EXP"": 1.0, ""EXR"": 1.0, ""F1"": 1.0, ""...",1.0,1.0,1.0
3,"{""EX"": 1, ""EXP"": 1.0, ""EXR"": 1.0, ""F1"": 1.0, ""...",1.0,1.0,1.0
4,"{""EXP"": 0.0, ""EXR"": 0.0, ""F1"": 0.0, ""EX"": 0, ""...",0.0,0.0,0.0


In [4]:
# Filter to keep question_ids where all three systems have EX==0
def get_valid_question_ids(df, metric_col):
    """Get question IDs where all systems have EX==0"""
    valid_qids = []

    for qid in df['question_id'].unique():
        q_data = df[df['question_id'] == qid]
        ex_values = q_data[f'{metric_col}_EX'].dropna()

        # Keep if all systems have EX==0
        if len(ex_values) > 0 and all(ex_values == 0):
            valid_qids.append(qid)

    return valid_qids

valid_qids = get_valid_question_ids(df, metric_col)
df_filtered = df[df['question_id'].isin(valid_qids)].copy()

print(f"Total questions: {df['question_id'].nunique()}")
print(f"Questions where all systems have EX==0: {len(valid_qids)}")
print(f"Filtered rows: {len(df_filtered)}")


Total questions: 1534
Questions where all systems have EX==0: 411
Filtered rows: 1232


In [27]:
# Find large gaps in EXP
def find_exp_gaps(df, metric_col, min_threshold=0.4, max_threshold=0.9):
    """Find question_ids with EXP gaps between min and max threshold"""
    results = []

    for qid in df['question_id'].unique():
        q_data = df[df['question_id'] == qid][
            ['system', 'db_name', 'question', 'gold_sql', 'predicted_sql',
             f'{metric_col}_EX', f'{metric_col}_EXP', f'{metric_col}_EXR']
        ].copy()

        if len(q_data) < 2:
            continue

        systems_list = q_data.to_dict('records')
        for sys1, sys2 in combinations(systems_list, 2):
            ex1 = sys1[f'{metric_col}_EX']
            ex2 = sys2[f'{metric_col}_EX']
            exp1 = sys1[f'{metric_col}_EXP']
            exp2 = sys2[f'{metric_col}_EXP']
            exr1 = sys1[f'{metric_col}_EXR']
            exr2 = sys2[f'{metric_col}_EXR']

            # Skip if any value is None
            if None in [ex1, ex2, exp1, exp2, exr1, exr2]:
                continue

            exp_gap = abs(exp1 - exp2)
            exr_gap = abs(exr1 - exr2)

            # Check if gap is within the specified range
            if min_threshold <= exp_gap <= max_threshold:
                results.append({
                    'question_id': qid,
                    'db_name': sys1['db_name'],
                    'question': sys1['question'],
                    'system1': sys1['system'],
                    'system2': sys2['system'],
                    'system1_EX': ex1,
                    'system2_EX': ex2,
                    'system1_EXP': exp1,
                    'system2_EXP': exp2,
                    'EXP_gap': exp_gap,
                    'system1_EXR': exr1,
                    'system2_EXR': exr2,
                    'EXR_gap': exr_gap,
                    'system1_sql': sys1['predicted_sql'],
                    'system2_sql': sys2['predicted_sql'],
                    'gold_sql': sys1['gold_sql']
                })

    return pd.DataFrame(results)

exp_gaps_df = find_exp_gaps(df_filtered, metric_col, min_threshold=0.4, max_threshold=0.95)
print(f"\nFound {len(exp_gaps_df)} cases with EXP gaps between 0.4 and 0.9")
exp_gaps_df.sort_values('EXP_gap', ascending=False)[
    ['question_id', 'system1', 'system2',
     'system1_EXP', 'system2_EXP', 'EXP_gap', 'system1_EXR', 'system2_EXR', 'EXR_gap']
]



Found 91 cases with EXP gaps between 0.4 and 0.9


,question_id,system1,system2,system1_EXP,system2_EXP,EXP_gap,system1_EXR,system2_EXR,EXR_gap
54,520,din-sql,mac-sql,0.071429,1.000000,0.928571,0.500000,0.000000,0.500000
53,520,chess,mac-sql,0.071429,1.000000,0.928571,0.500000,0.000000,0.500000
71,1064,chess,mac-sql,1.000000,0.071580,0.928420,0.500000,0.500000,0.000000
70,1064,chess,din-sql,1.000000,0.071580,0.928420,0.500000,0.500000,0.000000
21,145,din-sql,mac-sql,1.000000,0.111588,0.888412,0.019402,0.023194,0.003791
...,...,...,...,...,...,...,...,...,...
58,649,din-sql,mac-sql,0.500000,1.000000,0.500000,0.041667,0.041667,0.000000
59,682,chess,din-sql,0.500000,0.000000,0.500000,0.500000,0.000000,0.500000
78,1216,chess,din-sql,0.500000,0.016593,0.483407,1.000000,1.000000,0.000000
79,1216,chess,mac-sql,0.500000,0.033186,0.466814,1.000000,1.000000,0.000000


In [22]:
# Find large gaps in EXR
def find_exr_gaps(df, metric_col, min_threshold=0.4, max_threshold=0.9):
    """Find question_ids with EXR gaps between min and max threshold"""
    results = []

    for qid in df['question_id'].unique():
        q_data = df[df['question_id'] == qid][
            ['system', 'db_name', 'question', 'gold_sql', 'predicted_sql',
             f'{metric_col}_EX', f'{metric_col}_EXP', f'{metric_col}_EXR']
        ].copy()

        if len(q_data) < 2:
            continue

        systems_list = q_data.to_dict('records')
        for sys1, sys2 in combinations(systems_list, 2):
            ex1 = sys1[f'{metric_col}_EX']
            ex2 = sys2[f'{metric_col}_EX']
            exp1 = sys1[f'{metric_col}_EXP']
            exp2 = sys2[f'{metric_col}_EXP']
            exr1 = sys1[f'{metric_col}_EXR']
            exr2 = sys2[f'{metric_col}_EXR']

            # Skip if any value is None
            if None in [ex1, ex2, exp1, exp2, exr1, exr2]:
                continue

            exr_gap = abs(exr1 - exr2)
            exp_gap = abs(exp1 - exp2)

            # Check if gap is within the specified range
            if min_threshold <= exr_gap <= max_threshold:
                results.append({
                    'question_id': qid,
                    'db_name': sys1['db_name'],
                    'question': sys1['question'],
                    'system1': sys1['system'],
                    'system2': sys2['system'],
                    'system1_EX': ex1,
                    'system2_EX': ex2,
                    'system1_EXP': exp1,
                    'system2_EXP': exp2,
                    'EXP_gap': exp_gap,
                    'system1_EXR': exr1,
                    'system2_EXR': exr2,
                    'EXR_gap': exr_gap,
                    'system1_sql': sys1['predicted_sql'],
                    'system2_sql': sys2['predicted_sql'],
                    'gold_sql': sys1['gold_sql']
                })

    return pd.DataFrame(results)

exr_gaps_df = find_exr_gaps(df_filtered, metric_col, min_threshold=0.6, max_threshold=0.92)
print(f"\nFound {len(exr_gaps_df)} cases with EXR gaps between 0.4 and 0.9")
exr_gaps_df.sort_values('EXR_gap', ascending=False)[
    ['question_id', 'system1', 'system2', 'system1_EX', 'system2_EX',
     'system1_EXP', 'system2_EXP', 'EXP_gap', 'system1_EXR', 'system2_EXR', 'EXR_gap']
].head(20)



Found 17 cases with EXR gaps between 0.4 and 0.9


,question_id,system1,system2,system1_EX,system2_EX,system1_EXP,system2_EXP,EXP_gap,system1_EXR,system2_EXR,EXR_gap
7,720,chess,din-sql,0.0,0.0,0.000000,0.637255,0.637255,0.000000,0.915493,0.915493
8,720,chess,mac-sql,0.0,0.0,0.000000,0.719101,0.719101,0.000000,0.901408,0.901408
0,23,din-sql,mac-sql,0.0,0.0,0.392897,0.048426,0.344471,0.982244,0.096852,0.885391
1,41,chess,din-sql,0.0,0.0,1.000000,1.000000,0.000000,0.058824,0.911765,0.852941
9,894,chess,mac-sql,0.0,0.0,1.000000,0.000000,1.000000,0.750000,0.000000,0.750000
10,894,din-sql,mac-sql,0.0,0.0,1.000000,0.000000,1.000000,0.750000,0.000000,0.750000
2,41,chess,mac-sql,0.0,0.0,1.000000,0.500000,0.500000,0.058824,0.800000,0.741176
3,326,chess,mac-sql,0.0,0.0,1.000000,0.000000,1.000000,0.691176,0.000000,0.691176
5,442,chess,mac-sql,0.0,0.0,0.333333,0.333333,0.000000,1.000000,0.333333,0.666667
6,442,din-sql,mac-sql,0.0,0.0,0.333333,0.333333,0.000000,1.000000,0.333333,0.666667


In [29]:
# Interactive display function by system names and question_id
def display_comparison(df, question_id, system1_name, system2_name, metric_col='SEMANTIC_COLUMN_AND_EXACT_CELL'):
    """
    Display comparison for a specific question_id between two systems

    Parameters:
    - df: DataFrame with all data
    - question_id: The question ID to display
    - system1_name: Name of first system (e.g., 'chess', 'din-sql', 'mac-sql')
    - system2_name: Name of second system
    - metric_col: Metric column to display (default: SEMANTIC_COLUMN_AND_EXACT_CELL)
    """
    # Filter data for the specific question and systems
    q_data = df[df['question_id'] == question_id]
    sys1_data = q_data[q_data['system'] == system1_name]
    sys2_data = q_data[q_data['system'] == system2_name]

    if sys1_data.empty:
        print(f"No data found for system '{system1_name}' and question_id '{question_id}'")
        return
    if sys2_data.empty:
        print(f"No data found for system '{system2_name}' and question_id '{question_id}'")
        return

    sys1 = sys1_data.iloc[0]
    sys2 = sys2_data.iloc[0]

    # Extract metric values
    ex1 = sys1[f'{metric_col}_EX']
    ex2 = sys2[f'{metric_col}_EX']
    exp1 = sys1[f'{metric_col}_EXP']
    exp2 = sys2[f'{metric_col}_EXP']
    exr1 = sys1[f'{metric_col}_EXR']
    exr2 = sys2[f'{metric_col}_EXR']

    print("="*80)
    print(f"Question ID: {question_id} | Database: {sys1['db_name']}")
    print(f"\nQuestion: {sys1['question']}")

    # Print evidence if column exists
    if 'evidence' in sys1.index and pd.notna(sys1['evidence']):
        print(f"\nEvidence: {sys1['evidence']}")

    print(f"\n{system1_name} vs {system2_name}")
    print(f"  EX:  {ex1} vs {ex2}")
    if exp1 is not None and exp2 is not None:
        print(f"  EXP: {exp1:.3f} vs {exp2:.3f} (gap: {abs(exp1 - exp2):.3f})")
    else:
        print(f"  EXP: {exp1} vs {exp2}")
    if exr1 is not None and exr2 is not None:
        print(f"  EXR: {exr1:.3f} vs {exr2:.3f} (gap: {abs(exr1 - exr2):.3f})")
    else:
        print(f"  EXR: {exr1} vs {exr2}")
    print(f"\nGold SQL:\n{sys1['gold_sql']}")
    print(f"\n{system1_name} SQL:\n{sys1['predicted_sql']}")
    print(f"\n{system2_name} SQL:\n{sys2['predicted_sql']}")
    print("="*80 + "\n")

# Example usage
display_comparison(df, question_id=520, system1_name='din-sql', system2_name='mac-sql')


Question ID: 520 | Database: card_games

Question: Who is the illustrator that illustrated the least amount of cards? List the format of play of the cards that he/she illustrated.

din-sql vs mac-sql
  EX:  0.0 vs 0.0
  EXP: 0.071 vs 1.000 (gap: 0.929)
  EXR: 0.500 vs 0.000 (gap: 0.500)

Gold SQL:
SELECT T1.artist, T2.format FROM cards AS T1 INNER JOIN legalities AS T2 ON T2.uuid = T1.uuid GROUP BY T1.artist ORDER BY COUNT(T1.id) ASC LIMIT 1

din-sql SQL:
WITH artist_card_count AS ( SELECT artist, COUNT(*) AS card_count FROM cards GROUP BY artist ), least_artist AS ( SELECT artist FROM artist_card_count WHERE card_count = (SELECT MIN(card_count) FROM artist_card_count) ) SELECT DISTINCT l.format FROM cards c JOIN legalities l ON c.uuid = l.uuid WHERE c.artist IN (SELECT artist FROM least_artist);

mac-sql SQL:
SELECT DISTINCT T3.`format` FROM cards AS T1 INNER JOIN legalities AS T3 ON T1.`uuid` = T3.`uuid` WHERE T1.`artist` = ( SELECT T1.`artist` FROM cards AS T1 WHERE T1.`artist` IS N

In [ ]:
# Function to display all three systems for a question_id
def display_all_systems(df, question_id, metric_col='SEMANTIC_COLUMN_AND_EXACT_CELL'):
    """
    Display all systems' results for a specific question_id

    Parameters:
    - df: DataFrame with all data
    - question_id: The question ID to display
    - metric_col: Metric column to display
    """
    q_data = df[df['question_id'] == question_id]

    if q_data.empty:
        print(f"No data found for question_id '{question_id}'")
        return

    print("="*80)
    print(f"Question ID: {question_id} | Database: {q_data.iloc[0]['db_name']}")
    print(f"\nQuestion: {q_data.iloc[0]['question']}")
    print(f"\nGold SQL:\n{q_data.iloc[0]['gold_sql']}")
    print("\n" + "-"*80 + "\n")

    for _, row in q_data.iterrows():
        system = row['system']
        ex = row[f'{metric_col}_EX']
        exp = row[f'{metric_col}_EXP']
        exr = row[f'{metric_col}_EXR']

        print(f"System: {system}")
        print(f"  EX:  {ex}")
        print(f"  EXP: {exp:.3f}" if exp is not None else f"  EXP: {exp}")
        print(f"  EXR: {exr:.3f}" if exr is not None else f"  EXR: {exr}")
        print(f"  Predicted SQL:\n  {row['predicted_sql']}")
        print()

    print("="*80 + "\n")

# Example usage
# display_all_systems(df, question_id='some_id')


In [16]:
df[df['question_id'] == 23][['system', 'question', 'gold_sql',
       'predicted_sql','question_id', 'SEMANTIC_COLUMN_AND_EXACT_CELL']]


,system,question,gold_sql,predicted_sql,question_id,SEMANTIC_COLUMN_AND_EXACT_CELL
23,chess,List the names of schools with more than 30 di...,"SELECT T1.School, T1.Street FROM schools AS T1...","SELECT F.`School Name`, S.Street || ', ' || S....",23,"{""EX"": 0, ""EXP"": 0.0, ""EXR"": 0.0, ""F1"": 0.0, ""..."
1557,din-sql,List the names of schools with more than 30 di...,"SELECT T1.School, T1.Street FROM schools AS T1...","SELECT frpm.""School Name"", schools.""Street"", s...",23,"{""EX"": 0, ""EXP"": 0.39289749798224377, ""EXR"": 0..."
3091,mac-sql,List the names of schools with more than 30 di...,"SELECT T1.School, T1.StreetAbr FROM schools AS...","SELECT T2.`School`, T2.`Street`, T2.`City`, T2...",23,"{""EX"": 0, ""EXP"": 0.048426150121065374, ""EXR"": ..."
